# 04 — Baseline Citra: MobileNetV2

Tahap 5B. **Belum pernah dijalankan** di mesin penulis kode. Jalankan di mesin
compute.

Replikasi Tabel 1b paper (MobileNetV2 84,5% / 0,82) dan baris IMAGE Tabel 3.

**Prasyarat:** `01_data_audit.ipynb` sudah dijalankan (`manifest.csv` ada) dan
`02_text_features.ipynb` sudah membuat `split_seed*.json`.

Hyperparameter dari paper §4.2: SGD momentum 0,9, lr 0,01, batch 40, **200 epoch**.

> **Peringatan durasi.** MobileNetV2 pada 384x384 selama 200 epoch hampir pasti
> jauh di atas 15 menit. Jalankan sel estimasi dulu, laporkan angkanya, baru
> sepakati budget epoch sebenarnya.

In [ ]:
import sys, json, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src import datasets as DS
from src import splits as S
from src import train as T
from src.config import CONFIG, set_seed
from src.utils import save_fig

manifest = pd.read_csv(CONFIG["data_processed"] / "manifest.csv")
labels = manifest["kelas"].tolist()
y = DS.label_indices(labels)
device = T.pick_device()
print(f"{len(manifest)} dokumen | device: {device}")

## 1. Data citra

Resize 384x384 **tanpa padding** — aspect ratio sengaja di-warp, sesuai paper.
Grayscale diduplikasi 3 kanal agar bobot ImageNet berlaku.

Tidak ada augmentasi: paper mengukur augmentasi justru sedikit **menurunkan**
akurasi (83,9% dengan vs 84,5% tanpa), karena semua dokumen grayscale dengan
tulisan gelap di latar putih.

In [ ]:
from src.models import ImageBranch, build_transform

transform = build_transform()
set_seed(42)
split = S.load_split(42, labels)

def buat_dataset(idx):
    return DS.ImageDataset(manifest["path_citra"].tolist(), y, idx, transform)

loaders = DS.make_loaders(buat_dataset, split, num_workers=CONFIG["num_workers"])
xb, yb = next(iter(loaders["train"]))
print("batch citra:", tuple(xb.shape), "| label:", tuple(yb.shape))
assert xb.shape[1:] == (3, CONFIG["image_size"], CONFIG["image_size"])

## 2. Estimasi durasi

In [ ]:
est = T.estimate_duration(ImageBranch(), loaders, CONFIG["epochs"]["image"],
                          device=device)
print(json.dumps(est, indent=2))
print(f"\nPerkiraan {est['perkiraan_jam']} jam untuk {est['target_epochs']} epoch "
      f"per seed. Dikali 3 seed: {est['perkiraan_jam'] * 3:.1f} jam.")
print("\nKalau ini tidak masuk anggaran, turunkan EPOCHS di sel berikut dan catat "
      "sebagai deviasi eksplisit.")

In [ ]:
# Tidak ada yang perlu diatur di sini. Semuanya sudah dipilih otomatis:
# micro-batch dari VRAM, AMP dari jenis device, worker dari sistem operasi,
# epoch dan seed dari ada/tidaknya APDM_CEPAT.
from src.config import deviasi_runtime

EPOCHS = CONFIG["epochs"]["image"]
SEEDS = CONFIG["seeds"]
DEVIASI = deviasi_runtime()

print(f"{EPOCHS} epoch x {len(SEEDS)} seed | batch {CONFIG['batch_size']}"
      + (f" (micro-batch {CONFIG['micro_batch_size']})"
         if CONFIG["micro_batch_size"] else ""))
print("\nDeviasi yang dipaksa mesin ini — SALIN ke catatan eksperimen:")
for d in DEVIASI:
    print(f"  - {d}")
if not DEVIASI:
    print("  tidak ada; setelan persis seperti paper")

## 3. Training

Seluruh jaringan di-fine-tune; backbone **tidak** dibekukan. Paper melakukan
fine-tuning penuh, dan backbone beku akan menjatuhkan akurasi beberapa poin yang
kemudian tersalahartikan sebagai gap replikasi.

In [ ]:
def ringkas(nama, hasil_per_seed):
    """Cetak blok siap-salin untuk catatan eksperimen, satu baris per seed."""
    df = pd.DataFrame(hasil_per_seed)
    print(f"=== {nama} ===")
    print(df.to_string(index=False))
    print(f"\nOA       {df['oa'].mean():.4f} +/- {df['oa'].std(ddof=0):.4f}")
    print(f"macro F1 {df['macro_f1'].mean():.4f} +/- {df['macro_f1'].std(ddof=0):.4f}")
    print(f"\nUntuk frontmatter catatan (rata-rata 3 seed):")
    print(f"oa:: {df['oa'].mean():.4f}")
    print(f"macro_f1:: {df['macro_f1'].mean():.4f}")
    print(f"durasi:: {df['durasi_detik'].sum() / 60:.1f} menit total")
    return df

In [ ]:
hasil_img, pred_img = [], {}
for seed in SEEDS:
    set_seed(seed)
    sp = S.load_split(seed, labels)
    ld = DS.make_loaders(buat_dataset, sp, num_workers=CONFIG["num_workers"])
    model = ImageBranch(pretrained=True)
    assert all(p.requires_grad for p in model.trunk.parameters())

    print(f"\n[MobileNetV2 seed {seed}]")
    res = T.train_model(
        model, ld, epochs=EPOCHS, device=device,
        log_csv=CONFIG["root"] / "reports" / f"log_IMAGE_seed{seed}.csv",
        ckpt_path=CONFIG["models"] / f"IMAGE_seed{seed}.pt")
    ev = T.evaluate(model, ld["test"], device)

    hasil_img.append({"seed": seed, "oa": round(ev["oa"], 4),
                      "macro_f1": round(ev["macro_f1"], 4),
                      "epoch_terbaik": res["best_epoch"],
                      "epoch_dijalankan": res["epochs_run"],
                      "durasi_detik": res["durasi_detik"]})
    pred_img[seed] = ev

df_img = ringkas("MobileNetV2 (IMAGE)", hasil_img)

## 4. Perbandingan dengan paper

In [ ]:
print(f"OA kita  : {df_img['oa'].mean():.4f}   paper: 0.845")
print(f"F1 kita  : {df_img['macro_f1'].mean():.4f}   paper: 0.82")
print(f"selisih  : {df_img['oa'].mean() - 0.845:+.4f}")
print("\nPembanding lain di Tabel 3 paper: ensemble CNN Harley et al. 79,9%.")
print("MobileNetV2 tunggal mengalahkannya, jadi 84,5% bukan baseline lemah.")

fig = T.plot_confusion(pred_img[SEEDS[0]], f"MobileNetV2 (IMAGE) — seed {SEEDS[0]}")
save_fig(fig, f"05_confusion_image_seed{SEEDS[0]}")
print(T.report(pred_img[SEEDS[0]]))

In [ ]:
from sklearn.metrics import f1_score

per_kelas = pd.DataFrame(
    {seed: f1_score(e["y_true"], e["y_pred"], average=None,
                    labels=range(len(CONFIG["classes"])), zero_division=0)
     for seed, e in pred_img.items()}, index=CONFIG["classes"])
per_kelas["rata2"] = per_kelas.mean(axis=1)
per_kelas["paper_IMAGE"] = [0.94, 0.96, 0.85, 0.83, 0.90, 0.89, 0.83, 0.61, 0.80, 0.62]
display(per_kelas.round(3))

for seed, e in pred_img.items():
    np.save(CONFIG["data_processed"] / f"pred_image_seed{seed}.npy",
            np.stack([e["y_true"], e["y_pred"]]))
print("\nPrediksi IMAGE per seed disimpan — Tahap 6 membutuhkannya untuk oracle.")

## 5. Pratinjau oracle

Kalau `03_baseline_teks.ipynb` sudah dijalankan pada seed yang sama, oracle bisa
dihitung sekarang: satu sampel benar bila TEXT **atau** IMAGE benar. Ini batas atas
teoretis fusion, dan angkanya menentukan cara membaca hasil Tahap 6.

Paper: TEXT 73,8 / IMAGE 84,5 / FUSION 87,8 / **Oracle 92,1**. Artinya potensi di
atas baseline terbaik 7,6% absolut, dan fusion hanya memanen 3,3% — sekitar 43%.

In [ ]:
p_text = CONFIG["data_processed"] / "pred_text_seed42.npy"
p_img = CONFIG["data_processed"] / "pred_image_seed42.npy"

if p_text.exists() and p_img.exists():
    t_true, t_pred = np.load(p_text)
    i_true, i_pred = np.load(p_img)
    assert np.array_equal(t_true, i_true), \
        "urutan test berbeda antara TEXT dan IMAGE — oracle tidak sah"

    oa_t, oa_i = (t_pred == t_true).mean(), (i_pred == i_true).mean()
    # pred_b = IMAGE, baseline yang lebih kuat, dipakai sebagai tie-break
    orc = T.oracle(t_true, t_pred, i_pred)
    print(f"TEXT   {oa_t:.4f}")
    print(f"IMAGE  {oa_i:.4f}")
    print(f"Oracle {orc['oa']:.4f}  (macro F1 {orc['macro_f1']:.4f})")
    print(f"\npotensi di atas baseline terbaik: "
          f"{orc['oa'] - max(oa_t, oa_i):+.4f} absolut")
    print(f"keduanya benar : {orc['keduanya_benar']:.4f}")
    print(f"hanya TEXT     : {orc['hanya_a_benar']:.4f}")
    print(f"hanya IMAGE    : {orc['hanya_b_benar']:.4f}")
    print(f"keduanya salah : {orc['keduanya_salah']:.4f}")
    print("\n'hanya TEXT' adalah ukuran langsung komplementaritas: sampel yang "
          "citra gagal tapi teks berhasil.")
    print("'keduanya salah' adalah bagian yang TIDAK bisa diselamatkan skema "
          "fusion mana pun yang bekerja dengan memilih.")
else:
    print("Jalankan 03_baseline_teks.ipynb dulu untuk menghitung oracle.")

In [ ]:
print(f"\n----- IMAGE -----")
for _, r in df_img.iterrows():
    print(f"seed {int(r['seed'])}: oa:: {r['oa']}  macro_f1:: {r['macro_f1']}  "
          f"durasi:: {r['durasi_detik']}s")
print(f"rata-rata: oa:: {df_img['oa'].mean():.4f}  "
      f"macro_f1:: {df_img['macro_f1'].mean():.4f}")
print("DEVIASI:", DEVIASI or "tidak ada")